# 🧩 Mini-Lab: Few-Shot Examples

**Module 4: Prompt Engineering & Security** | **Duration: ~30 min** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Compare** zero-shot and few-shot prompting and understand when each is appropriate
2. **Apply** few-shot prompting by providing examples that guide the model's output
3. **Build** reusable prompt templates that inject examples and context dynamically
4. **Recognize** how context injection provides the model with external information at prompt time

## Target Concepts

| Concept | Description |
|---------|-------------|
| Zero-Shot Prompting | Asking the model to perform a task with no examples — the baseline approach |
| Few-Shot Prompting | Providing examples in the prompt so the model learns the desired pattern in-context |
| Prompt Templates | Reusable prompt structures with placeholders for dynamic content |
| Context Injection | Adding external information (examples, data, context) into prompts at runtime |

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)
# MODEL = "deepseek-r1:1.5b"
MODEL = "llama3.2:3b"
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def chat(messages, model=MODEL):
    """Send messages to the Chat Completions API and return the response text."""
    response = client.chat.completions.create(model=model, messages=messages, temperature=0)
    return response.choices[0].message.content

## 1. Zero-Shot Prompting — The Baseline

In **zero-shot prompting**, we simply ask the model to perform a task with no examples. The model relies entirely on its pre-training to figure out what we want.

Let's try a sentiment classification task with zero-shot:

In [40]:
zero_shot_messages = [
    {"role": "system", "content": "Classify the sentiment of the text as: positive, negative, or neutral. Reply with only the label."},
    {"role": "user", "content": "The battery lasts all day, but the screen is dim and hard to read outdoors."}
]

result = chat(zero_shot_messages)
print(f"Zero-shot result: {result}")

Zero-shot result: neutral


Zero-shot works well for straightforward tasks. But what if the model's interpretation of "mixed" vs. "neutral" doesn't match ours? We have no way to steer it without examples.

Let's try a harder task — classifying customer support tickets into custom categories:

In [41]:
zero_shot_ticket = [
    {"role": "system", "content": "Classify the support ticket into one of these categories: billing, technical, account, shipping. Reply with only the category."},
    {"role": "user", "content": "I was charged twice for my last order and need a refund for the duplicate payment."}
]

result = chat(zero_shot_ticket)
print(f"Zero-shot ticket classification: {result}")

Zero-shot ticket classification: billing


This probably works fine for clear-cut cases. But for ambiguous tickets, the model may guess differently than we'd like. **Few-shot prompting** solves this by showing the model what we expect.

## 2. Few-Shot Prompting — Teaching by Example

In **few-shot prompting**, we include examples of input → output pairs directly in the prompt. The model learns the pattern from these examples and applies it to new inputs.

Let's redo the ticket classification, but this time with examples:

In [42]:
few_shot_messages = [
    {"role": "system", "content": "Classify the support ticket into one of these categories: billing, technical, account, shipping. Reply with only the category."},
    # Example 1
    {"role": "user", "content": "My package hasn't arrived and the tracking shows it's stuck in transit."},
    {"role": "assistant", "content": "shipping"},
    # Example 2
    {"role": "user", "content": "I can't log into my account, it says my password is wrong but I just reset it."},
    {"role": "assistant", "content": "account"},
    # Example 3
    {"role": "user", "content": "The app crashes every time I try to upload a photo."},
    {"role": "assistant", "content": "technical"},
    # Example 4
    {"role": "user", "content": "I see an extra duplicate charge on my credit card statement from your company, and my refund was not processed."},
    {"role": "assistant", "content": "billing"},
    # Now the real ticket
    {"role": "user", "content": "I was charged twice for my last order and need a refund for the duplicate payment."}
]

result = chat(few_shot_messages)
print(f"Few-shot ticket classification: {result}")

Few-shot ticket classification: billing


Notice how we used `user` / `assistant` message pairs to provide examples. The model sees the pattern and applies it consistently.

Let's test with an **ambiguous** ticket to see few-shot's advantage:

In [52]:
ambiguous_ticket = (
    "I shipped my return two weeks ago and tracking confirms delivery "
    "to your warehouse, but the refund still hasn't appeared on my credit card."
)

# Replace only the last user message with the ambiguous ticket
ambiguous_messages = few_shot_messages[:-1] + [
    {"role": "user", "content": ambiguous_ticket}
]

result_few = chat(ambiguous_messages)

# Compare with zero-shot on the same ambiguous ticket
zero_shot_ambiguous = [
    {"role": "system", "content": "Classify the support ticket into one of these categories: technical, account, billing, shipping. Reply with only the category."},
    {"role": "user", "content": ambiguous_ticket}
]
result_zero = chat(zero_shot_ambiguous)

md(f"**Ambiguous ticket:** *{ambiguous_ticket}*\n\n"
   f"| Approach | Classification |\n|----------|----------------|\n"
   f"| Zero-shot | {result_zero} |\n"
   f"| Few-shot | {result_few} |")

**Ambiguous ticket:** *I shipped my return two weeks ago and tracking confirms delivery to your warehouse, but the refund still hasn't appeared on my credit card.*

| Approach | Classification |
|----------|----------------|
| Zero-shot | account |
| Few-shot | billing |

The zero-shot approach returned a single category but picked the wrong class — it latched onto surface-level keywords like "shipped" and "tracking." The few-shot version correctly identified this as **billing** because our examples taught the model that refund and charge issues go to billing, regardless of other topics mentioned. **Few-shot encodes your team's decision-making logic through examples, not just instructions.**

## 3. Prompt Templates — Reusable Patterns

Hard-coding examples into every prompt isn't practical. **Prompt templates** let us define a reusable structure with placeholders that we fill in dynamically.

Let's create a template for our few-shot classifier:

In [56]:
# Define our examples as data
examples = [
    {"text": "My package hasn't arrived and the tracking shows it's stuck in transit.", "label": "shipping"},
    {"text": "I can't log into my account, it says my password is wrong but I just reset it.", "label": "account"},
    {"text": "The app crashes every time I try to upload a photo.", "label": "technical"},
    {"text": "I see an extra charge on my credit card statement from your company.", "label": "billing"},
]

client = OpenAI()
MODEL = "gpt-4o-mini"

def chat(messages, model=MODEL):
    """Send messages to the Chat Completions API and return the response text."""
    response = client.chat.completions.create(model=model, messages=messages, temperature=0)
    return response.choices[0].message.content

def build_few_shot_messages(system_prompt, examples, new_input):
    """Build a few-shot message list from a template."""
    messages = [{"role": "system", "content": system_prompt}]
    
    # Inject examples as user/assistant pairs
    for ex in examples:
        messages.append({"role": "user", "content": ex["text"]})
        messages.append({"role": "assistant", "content": ex["label"]})
    
    # Add the new input
    messages.append({"role": "user", "content": new_input})
    return messages

# Use the template
system = "Classify the support ticket into one of these categories: billing, technical, account, shipping. Reply with only the category."

test_tickets = [
    "The website keeps showing a 500 error when I try to checkout.",
    "I need to update my email address on my profile.",
    "Where is my refund? It's been two weeks since I returned the item.",
]

for ticket in test_tickets:
    messages = build_few_shot_messages(system, examples, ticket)
    result = chat(messages)
    print(f"{result:10s} ← {ticket}")

technical  ← The website keeps showing a 500 error when I try to checkout.
account    ← I need to update my email address on my profile.
billing    ← Where is my refund? It's been two weeks since I returned the item.


The template function `build_few_shot_messages` separates our **prompt structure** from our **data**, making it easy to:
- Swap in different example sets
- Process many inputs with the same pattern
- Maintain and version the template independently

## 4. Context Injection — Adding External Information

**Context injection** means inserting external information into a prompt at runtime. The few-shot examples above are one form of context injection. But context injection goes beyond examples — you can inject reference data, domain knowledge, or retrieved documents.

Let's build a template that injects a product catalog so the model can answer questions about products it was never trained on:

In [57]:
# Simulated product catalog (external context)
product_catalog = """Product Catalog:
- WidgetPro X1: Smart home controller, $149, supports Zigbee and Z-Wave, 2-year warranty
- WidgetPro X2: Smart home controller, $249, adds Matter support and touchscreen, 3-year warranty
- SensorPack S1: 4 motion sensors + 2 door sensors, $79, requires WidgetPro hub
- CamView C1: Indoor security camera, $59, 1080p, night vision, cloud storage $3/mo"""

def build_context_prompt(context, question):
    """Build a prompt that injects external context for the model to reference."""
    system_msg = (
        "You are a helpful product support assistant. "
        "Answer questions using ONLY the product catalog provided below. "
        "If the answer isn't in the catalog, say so.\n\n"
        f"{context}"
    )
    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": question}
    ]

# Ask a question that requires the injected context
question = "What's the difference between the X1 and X2? Is the upgrade worth it?"
messages = build_context_prompt(product_catalog, question)

result = chat(messages)
md(result)

The difference between the WidgetPro X1 and X2 is that the X2 adds Matter support and features a touchscreen, while the X1 supports Zigbee and Z-Wave. Additionally, the X1 comes with a 2-year warranty, whereas the X2 has a 3-year warranty. Whether the upgrade is worth it depends on your specific needs for Matter support and a touchscreen interface.

The model has no training data about "WidgetPro" — it answered entirely from the **injected context**. This is the same principle behind RAG (Retrieval-Augmented Generation), which you'll explore in a later module.

## 5. Combining It All — Templated Few-Shot with Context

Let's combine few-shot examples, prompt templates, and context injection into a single reusable pattern. We'll build a product review summarizer that follows a specific output format:

In [58]:
# Few-shot examples define the desired output format
summary_examples = [
    {
        "review": "Great camera but the app is buggy. Picture quality is amazing at night. Setup took forever.",
        "summary": "👍 Pros: Excellent night picture quality\n👎 Cons: Buggy app, difficult setup\n⭐ Overall: Mixed"
    },
    {
        "review": "Fast delivery, works perfectly with my smart home. Love the touchscreen. A bit pricey though.",
        "summary": "👍 Pros: Fast delivery, seamless smart home integration, great touchscreen\n👎 Cons: High price\n⭐ Overall: Positive"
    }
]

def build_review_summarizer(examples, new_review):
    """Few-shot template for review summarization with a specific format."""
    messages = [
        {"role": "system", "content": "Summarize product reviews using the exact format shown in the examples."}
    ]
    for ex in examples:
        messages.append({"role": "user", "content": ex["review"]})
        messages.append({"role": "assistant", "content": ex["summary"]})
    messages.append({"role": "user", "content": new_review})
    return messages

# Test with a new review
new_review = (
    "The motion sensors are super responsive and battery lasts months. "
    "However, they only work with the WidgetPro hub which I had to buy separately. "
    "Installation was straightforward."
)

messages = build_review_summarizer(summary_examples, new_review)
result = chat(messages)
md(result)

👍 Pros: Highly responsive motion sensors, long battery life, easy installation
👎 Cons: Requires separate purchase of WidgetPro hub
⭐ Overall: Positive

The model follows our custom emoji-based format perfectly — it learned the pattern from just two examples. Without those examples (zero-shot), the model would use its own format, which might vary unpredictably.

## 🎯 Summary

### Key Takeaways

1. **Zero-shot prompting** — asks the model to perform a task with no examples; works for straightforward tasks but offers no control over output patterns
2. **Few-shot prompting** — provides input/output examples in the prompt so the model learns the desired pattern in-context, improving consistency and accuracy
3. **Prompt templates** — separate prompt structure from data, enabling reusable patterns that can be filled dynamically at runtime
4. **Context injection** — inserts external information (catalogs, documents, examples) into prompts, letting the model answer based on data it was never trained on